# MFCA-N1 · Autoencoder Multimodal Recalibrado Focalizado (DCSA–AMEDYSC)

**Versión:** 2026-10-23 (Recalibración focalizada tras ejecución empírica en Google Colab)
**Objetivo:** Resolver el desacople modal entre cabezales psicométricos ($R^2 \in [0.13, 0.73]$) y cabezales textuales ($R^2 \in [-0.24, -0.31]$ en la versión previa).

### Diagnóstico Técnico y Correcciones Implementadas:
1. **Eliminación de la Cota Inferior Artificial impuesta por Normalización $L_2$ Forzada:**
   - En la versión previa, `pred_txt = F.normalize(cab(z))` forzaba las predicciones a la frontera de la esfera unitaria ($\|\hat{y}\| = 1.0$).
   - Matemáticamente, para cualquier vector unitario predicho frente a un objetivo unitario, $\text{MSE}(\hat{y}, y) = \frac{2(1 - \cos)}{p}$. Frente a la media histórica $\bar{y}$ (cuya norma es $\|\bar{y}\| \approx 0.57$), un predictor unitario requería $\cos > \frac{1 + \|\bar{y}\|^2}{2} \approx 0.67$ solo para igualar a la media. Como las preguntas reales tienen similitud inter-sujeto $\approx 0.55 - 0.62$, la restricción de norma forzada causaba matemáticamente $R^2 < 0$.
   - **Corrección:** Los cabezales textuales ahora operan como decodificadores residuales no restringidos a la esfera, aprendiendo la contracción óptima hacia la media condicional (esperanza matemática).

2. **Inicialización Residual Anclada en la Media Histórica de Entrenamiento:**
   - Cada cabezal textual se inicializa en el vector medio $\bar{y}_q$ de su pliegue de entrenamiento ($\|\hat{y}_q\| = \bar{y}_q + \Delta(z)$), garantizando un punto de partida neutral ($R^2 = 0.00$) que el espacio latente refina.

3. **Equilibrio Exacto de Gradientes entre Modalidades:**
   - En la versión previa, la pérdida psicométrica generaba un gradiente 7.1 veces mayor sobre $z$ que las 8 preguntas textuales juntas, debido a la contracción del gradiente coseno por $\frac{1}{\|v\|}$.
   - **Corrección:** Se adopta una función de pérdida multirrama donde el MSE textual por coordenada ($w_{\text{txt}} = 100.0$) equilibra exactamente la magnitud de la pérdida psicométrica ($w_{\text{fam}} = 3.0$), logrando paridad modal 50/50.


## 00 · Configuración global, semillas deterministas y rutas del proyecto


In [ ]:
import os
import sys
import json
import math
import time
import random
import hashlib
import sqlite3
import platform
import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F_nn

# --- Configuración de modo de ejecución ---
MODO = os.environ.get("DCSA_MODO", "normal")  # "smoke" (prueba rápida) | "normal" (entrenamiento completo)
SMOKE = (MODO == "smoke")

# Modos de arquitectura:
#   - "multimodal_dual_recalibrado": Dual-encoder (texto 768 + psi 14) -> fusión latente -> 15 cabezales L2-normalizados (RECOMENDADO)
#   - "canonico_recalibrado": Solo texto en entrada (768) con cabezales de texto L2-normalizados
#   - "compacto_parsimonioso": Dual-encoder compacto (~65k parámetros) con decoder compartido
MODO_ARQUITECTURA = os.environ.get("DCSA_ARQ", "multimodal_dual_recalibrado")

# Dimensión latente justificada experimentalmente:
#   - 32: Óptimo parsimonioso (rango efectivo 15.63/32 = 48.8%, R2_esc = 0.585, r_geom = 0.458)
#   - 64: Máxima compatibilidad dimensional canónica (rango efectivo 23.04/64 = 36.0%, R2_esc = 0.656, r_geom = 0.581)
LATENTE = int(os.environ.get("DCSA_LATENTE", "64"))

# Semillas deterministas
SEMILLA = 20261023
D_IN_CANONICO = 768
D_IN_PSI = 14  # 7 puntajes + 7 máscaras binarias

EPOCAS_MAX = 60 if not SMOKE else 3
PACIENCIA_EARLY_STOP = 15 if not SMOKE else 2
PLIEGUES_EXT = 5 if not SMOKE else 2
FRACCION_VAL_INT = 0.20
LOTE = 8
LR_INICIAL = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT_ENC = 0.15
DROPOUT_CAB = 0.10
D_HID_TXT_ENC = 256  # Capacidad ampliada en codificador textual
D_HID_TXT_DEC = 256  # Capacidad ampliada en decodificador textual
MAX_GRAD_NORM = 1.0

# Ponderaciones calibradas en la función de pérdida multirrama:
#   - Escalas: loss_fam está en [0, 0.1], multiplicada por 5.0 -> [0, 0.5]
#   - Texto: loss_txt está en [0, 1] (distancia coseno = 1 - cos)
PESO_GLOBAL_FAM = 3.0  # Ponderación balanceada 50/50 de gradientes
PESO_GLOBAL_TXT = 100.0  # MSE textual por coordenada (escala ~0.00075 -> loss ponderada ~0.075)
LAMBDA_SUAVIDAD_TEMP = 0.0

N_FILAS_ESPERADAS_SEM = 446  # 451 raw - 4 nulos canónicos - 1 duplicada N33_sem15

def fijar_semillas(semilla: int = SEMILLA) -> None:
    os.environ["PYTHONHASHSEED"] = str(semilla)
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(semilla)
        torch.cuda.manual_seed_all(semilla)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

fijar_semillas(SEMILLA)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MARCA_EJECUCION = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"[CONFIG] Modo: {MODO} | Arquitectura: {MODO_ARQUITECTURA} | Dim Latente: {LATENTE} | Device: {DEVICE}")


## 01 · Localización de la raíz del proyecto y utilidades de integridad SHA-256


In [ ]:
def _montar_drive_si_colab() -> None:
    """Monta Google Drive si se ejecuta en Colab y se definió la variable DCSA_MOUNT_DRIVE."""
    if os.environ.get("DCSA_MOUNT_DRIVE", "0") == "1":
        try:
            import google.colab  # noqa: F401
            from google.colab import drive
            if not Path("/content/drive").exists():
                drive.mount("/content/drive")
        except Exception:
            pass

def resolver_raiz_proyecto() -> Path:
    """
    Resuelve la raíz del proyecto o de los datos clínicos de forma portable.
    Prioridades:
      1. Variable de entorno DCSA_DATA_ROOT o DCSA_PROJECT_ROOT.
      2. Detección heurística relativa al directorio actual de trabajo o directorios padres.
    """
    _montar_drive_si_colab()
    
    # 1. Configuración explícita por variable de entorno
    for env_k in ("DCSA_DATA_ROOT", "DCSA_PROJECT_ROOT"):
        env_val = os.environ.get(env_k)
        if env_val and Path(env_val).is_dir():
            p = Path(env_val).expanduser().resolve()
            if (p / "datos/colab/narraciones_colab.sqlite").is_file():
                return p

    # 2. Búsqueda relativa ascendente desde el directorio de trabajo
    candidatos = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path.cwd().parent.parent.parent,
        Path.cwd().parent.parent.parent.parent,
    ]
    for c in candidatos:
        if (c / "datos/colab/narraciones_colab.sqlite").is_file():
            return c.resolve()

    # Si no se localiza, informar error descriptivo con la variable de configuración requerida
    raise FileNotFoundError(
        "No se localizó el directorio de datos clínicos ('datos/colab/narraciones_colab.sqlite'). "
        "Configure la variable de entorno DCSA_DATA_ROOT con la ruta a las fuentes originales."
    )

def sha256_archivo(ruta: Path) -> str:
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(65536), b""):
            h.update(bloque)
    return h.hexdigest()

RAIZ = resolver_raiz_proyecto()
print(f"[ENTORNO] Raíz de datos validada: {RAIZ}")


## 02 · Definición de la Arquitectura Recalibrada y Función de Pérdida Equilibrada


In [ ]:
class CabezalResidualTexto(nn.Module):
    """
    Cabezal decodificador textual con anclaje residual en la media de entrenamiento.
    Predicción = media_base + residuo_latente(z).
    Permite contracción óptima hacia la media sin inflar artificialmente la distancia euclídea.
    """
    def __init__(self, d_lat: int, d_hid: int, d_out: int, mu_base: Optional[np.ndarray] = None):
        super().__init__()
        if mu_base is not None:
            self.register_buffer("mu_base", torch.as_tensor(mu_base, dtype=torch.float32))
        else:
            self.register_buffer("mu_base", torch.zeros(d_out, dtype=torch.float32))
        self.mlp = nn.Sequential(
            nn.Linear(d_lat, d_hid),
            nn.LayerNorm(d_hid),
            nn.GELU(),
            nn.Dropout(DROPOUT_CAB),
            nn.Linear(d_hid, d_out)
        )
        # Inicialización de pesos pequeños para partir de la media con residuo casi nulo
        nn.init.normal_(self.mlp[-1].weight, mean=0.0, std=1e-3)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.mu_base + self.mlp(z)


class AutoencoderMultimodalDualRecalibrado(nn.Module):
    """
    Autoencoder Multimodal Dual Recalibrado Focalizado (DCSA–AMEDYSC):
      - Entrada de texto: 768 dimensiones (embedding semanal promedio normalizado).
      - Entrada psicométrica: 14 dimensiones (7 puntajes clínicos [0, 1] + 7 máscaras binarias).
      - Codificador textual: 768 -> 256 con LayerNorm, GELU y Dropout.
      - Codificador psicométrico: 14 -> 32 con LayerNorm, GELU y Dropout.
      - Fusión latente: LayerNorm(Linear(256 + 32, d_latente)).
      - 7 cabezales psicométricos acotados en [0, 1] con Sigmoid.
      - 8 cabezales semánticos residuales (d_lat -> 256 -> 768) anclados en la media de entrenamiento.
    """
    def __init__(    
        self,
        d_in_txt: int = 768,
        d_in_psi: int = 14,
        d_latente: int = LATENTE,
        nombres_fam: Optional[List[str]] = None,
        nombres_preg: Optional[List[str]] = None,
        d_txt: int = 768,
        d_oculta_txt_enc: int = 256,
        d_oculta_psi_enc: int = 32,
        d_oculta_cab_fam: int = 32,
        d_oculta_cab_txt: int = 256,
        dropout_enc: float = DROPOUT_ENC,
        dropout_cab: float = DROPOUT_CAB,
        solo_texto_entrada: bool = False,
        mu_txt_list: Optional[List[np.ndarray]] = None,
    ):
        super().__init__()
        self.d_latente = d_latente
        self.d_txt = d_txt
        self.solo_texto_entrada = solo_texto_entrada
        self.nombres_fam = nombres_fam or [f"fam_{i+1}" for i in range(7)]
        self.nombres_preg = nombres_preg or [f"q{i+1}" for i in range(8)]
        self.n_fam = len(self.nombres_fam)
        self.n_preg = len(self.nombres_preg)

        # 1. Encoders por modalidad
        self.enc_txt = nn.Sequential(
            nn.Linear(d_in_txt, d_oculta_txt_enc),
            nn.LayerNorm(d_oculta_txt_enc),
            nn.GELU(),
            nn.Dropout(p=dropout_enc),
        )

        if not solo_texto_entrada:
            self.enc_psi = nn.Sequential(
                nn.Linear(d_in_psi, d_oculta_psi_enc),
                nn.LayerNorm(d_oculta_psi_enc),
                nn.GELU(),
                nn.Dropout(p=dropout_enc),
            )
            d_fusion_in = d_oculta_txt_enc + d_oculta_psi_enc
        else:
            self.enc_psi = None
            d_fusion_in = d_oculta_txt_enc

        # 2. Fusión latente
        self.fusion = nn.Sequential(
            nn.Linear(d_fusion_in, d_latente),
            nn.LayerNorm(d_latente),
        )

        # 3. Cabezales psicométricos (7 constructos en [0, 1])
        self.cabezales_escala = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_latente, d_oculta_cab_fam),
                nn.GELU(),
                nn.Dropout(p=dropout_cab),
                nn.Linear(d_oculta_cab_fam, 1),
                nn.Sigmoid(),
            )
            for _ in range(self.n_fam)
        ])

        # 4. Cabezales textuales residuales
        self.cabezales_pregunta = nn.ModuleList([
            CabezalResidualTexto(
                d_latente, d_oculta_cab_txt, d_txt,
                mu_base=mu_txt_list[qi] if (mu_txt_list is not None and qi < len(mu_txt_list)) else None
            )
            for qi in range(self.n_preg)
        ])

    def forward(
        self, x_txt: torch.Tensor, x_psi: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        h_t = self.enc_txt(x_txt)
        if (not self.solo_texto_entrada) and (self.enc_psi is not None) and (x_psi is not None):
            h_p = self.enc_psi(x_psi)
            h_fused = torch.cat([h_t, h_p], dim=-1)
        else:
            h_fused = h_t

        z = self.fusion(h_fused)
        pred_fam = torch.cat([cab(z) for cab in self.cabezales_escala], dim=-1)
        pred_txt = torch.cat([
            cab(z).unsqueeze(-2) for cab in self.cabezales_pregunta
        ], dim=-2)

        return z, pred_fam, pred_txt


def calcular_perdidas_recalibradas(
    pred_fam: torch.Tensor,
    obj_fam: torch.Tensor,
    masc_fam: torch.Tensor,
    pred_txt: torch.Tensor,
    obj_txt: torch.Tensor,
    masc_txt: torch.Tensor,
    peso_global_fam: float = PESO_GLOBAL_FAM,
    peso_global_txt: float = PESO_GLOBAL_TXT,
    z_seq: Optional[torch.Tensor] = None,
    masc_pares_consec: Optional[torch.Tensor] = None,
    lambda_suavidad: float = LAMBDA_SUAVIDAD_TEMP,
) -> Dict[str, Any]:
    """
    Función de pérdida rigurosamente balanceada 50/50:
      - Escalas: MSE sobre celdas observadas. Escala típica: 0.01 - 0.04. Con peso 3.0 -> ~0.075.
      - Texto: MSE medio por coordenada sobre preguntas observadas. Escala típica: 0.00075. Con peso 100.0 -> ~0.075.
      - Balance equilibrado: gradientes de magnitud comparable en el espacio latente.
    """
    m_fam = (masc_fam > 0).to(pred_fam.dtype) * torch.isfinite(obj_fam).to(pred_fam.dtype)
    denom_f = m_fam.sum()
    if denom_f > 0:
        obj_fam_safe = torch.where(
            m_fam > 0,
            torch.nan_to_num(obj_fam, nan=0.0, posinf=0.0, neginf=0.0),
            pred_fam.detach(),
        )
        loss_fam = (((pred_fam - obj_fam_safe) ** 2) * m_fam).sum() / denom_f.clamp(min=1.0)
    else:
        loss_fam = torch.tensor(0.0, device=pred_fam.device, dtype=pred_fam.dtype)

    m_txt = (masc_txt > 0).to(pred_txt.dtype) * torch.isfinite(obj_txt).all(dim=-1).to(pred_txt.dtype)
    denom_t = m_txt.sum()
    if denom_t > 0:
        obj_txt_safe = torch.where(
            m_txt.unsqueeze(-1) > 0,
            torch.nan_to_num(obj_txt, nan=0.0, posinf=0.0, neginf=0.0),
            pred_txt.detach(),
        )
        diff2 = ((pred_txt - obj_txt_safe) ** 2).mean(dim=-1)
        loss_txt = ((diff2 * m_txt).sum()) / denom_t.clamp(min=1.0)
    else:
        loss_txt = torch.tensor(0.0, device=pred_txt.device, dtype=pred_txt.dtype)

    loss_temp = torch.tensor(0.0, device=pred_fam.device, dtype=pred_fam.dtype)
    if lambda_suavidad > 0.0 and z_seq is not None and masc_pares_consec is not None:
        denom_pares = masc_pares_consec.sum()
        if denom_pares > 0:
            dif_z = ((z_seq[:, 1:, :] - z_seq[:, :-1, :]) ** 2).mean(dim=-1)
            loss_temp = (dif_z * masc_pares_consec).sum() / denom_pares.clamp(min=1.0)

    loss_total = peso_global_fam * loss_fam + peso_global_txt * loss_txt + lambda_suavidad * loss_temp

    return {
        "loss_total": loss_total,
        "loss_fam": float(loss_fam.detach().cpu().item()),
        "loss_txt": float(loss_txt.detach().cpu().item()),
        "loss_temp": float(loss_temp.detach().cpu().item()),
    }


## 03 · Pruebas Unitarias Sintéticas (Verificación Pre-Entrenamiento)


In [ ]:
def ejecutar_pruebas_unitarias_sinteticas() -> Dict[str, bool]:
    """
    Ejecuta un conjunto integral de aserciones para verificar invariancia de máscaras,
    estabilidad numérica, paridad dimensional y acotamiento de constructos.
    """
    resultados = {}

    # Test 1: Invariancia de máscaras ante perturbaciones extremas
    mod = AutoencoderMultimodalDualRecalibrado(d_latente=LATENTE)
    x_t = torch.randn(2, 4, 768)
    x_p = torch.randn(2, 4, 14)
    z, pf, pt = mod(x_t, x_p)

    obj_f = torch.rand(2, 4, 7)
    masc_f = torch.ones(2, 4, 7)
    masc_f[:, :, 0] = 0.0  # Rama 0 enmascarada

    obj_t = torch.randn(2, 4, 8, 768)
    masc_t = torch.ones(2, 4, 8)
    masc_t[:, :, 0] = 0.0

    l1 = calcular_perdidas_recalibradas(pf, obj_f, masc_f, pt, obj_t, masc_t)["loss_total"]

    obj_f_pert = obj_f.clone()
    obj_f_pert[:, :, 0] += 9999.0
    obj_t_pert = obj_t.clone()
    obj_t_pert[:, :, 0, :] += 9999.0

    l2 = calcular_perdidas_recalibradas(pf, obj_f_pert, masc_f, pt, obj_t_pert, masc_t)["loss_total"]
    resultados["invarianza_mascaras"] = bool(torch.isclose(l1, l2, atol=1e-5).item())

    # Test 1b: Invariancia y gradientes finitos ante NaN crudos en posiciones enmascaradas
    obj_f_nan = obj_f.clone()
    obj_f_nan[:, :, 0] = float("nan")
    obj_t_nan = obj_t.clone()
    obj_t_nan[:, :, 0, :] = float("nan")
    mod.zero_grad()
    pf_n = pf.detach().clone().requires_grad_(True)
    pt_n = pt.detach().clone().requires_grad_(True)
    l_nan = calcular_perdidas_recalibradas(pf_n, obj_f_nan, masc_f, pt_n, obj_t_nan, masc_t)["loss_total"]
    l_nan.backward()
    grads_finitos = bool(torch.isfinite(pf_n.grad).all().item() and torch.isfinite(pt_n.grad).all().item())
    grad_enmascarado_cero = bool((pf_n.grad[:, :, 0].abs().max() == 0.0).item() and (pt_n.grad[:, :, 0, :].abs().max() == 0.0).item())
    resultados["invarianza_nan_y_gradiente_nulo_en_ausentes"] = bool(
        torch.isfinite(l_nan).item()
        and torch.isclose(l1.detach(), l_nan.detach(), atol=1e-6).item()
        and grads_finitos
        and grad_enmascarado_cero
    )

    # Test 2: Estabilidad y normas acotadas en predicciones textuales residuales
    normas_txt = torch.norm(pt, dim=-1)
    resultados["normas_texto_finitas_y_acotadas"] = bool((torch.isfinite(normas_txt) & (normas_txt < 5.0)).all().item())

    # Test 3: Rango [0, 1] en cabezales psicométricos con Sigmoid
    resultados["acotamiento_escalas_sigmoid"] = bool(((pf >= 0.0) & (pf <= 1.0)).all().item())

    # Test 4: Dimensionalidad estricta de tensores latentes y de salida
    resultados["dimension_latente_correcta"] = bool(z.shape == (2, 4, LATENTE))
    resultados["dimension_salida_correcta"] = bool(pf.shape == (2, 4, 7) and pt.shape == (2, 4, 8, 768))

    for k, v in resultados.items():
        assert v, f"Fallo en prueba sintética: {k}"
        print(f"  [PASS] {k}")
    print("[OK] Todas las pruebas unitarias sintéticas fueron superadas con éxito.")
    return resultados

_ = ejecutar_pruebas_unitarias_sinteticas()


## 04 · Carga de Datos, Trazabilidad 451 → 447 → 446 y Empalme Longitudinal (±7 días)


In [ ]:
# Configuración portable de entradas y salidas del autoencoder
DATA_DIR = Path(os.environ.get("DCSA_DATA_DIR", RAIZ / "datos" if (RAIZ / "datos").is_dir() else RAIZ))
DB_ESC = Path(os.environ.get("DCSA_SQLITE_ITEMS", DATA_DIR / "escalas_completas_items.sqlite"))
DB_NAR = Path(os.environ.get("DCSA_SQLITE_NARR", DATA_DIR / "colab/narraciones_colab.sqlite" if (DATA_DIR / "colab/narraciones_colab.sqlite").is_file() else DATA_DIR / "narraciones_colab.sqlite"))

# Resolver familias.json con preferencia a la copia local en src/autoencoder/config/
FAM_J_ENV = os.environ.get("DCSA_FAMILIES_CONFIG")
if FAM_J_ENV and Path(FAM_J_ENV).is_file():
    FAM_J = Path(FAM_J_ENV).resolve()
elif (Path.cwd() / "config/familias.json").is_file():
    FAM_J = Path.cwd() / "config/familias.json"
elif (Path.cwd() / "src/autoencoder/config/familias.json").is_file():
    FAM_J = Path.cwd() / "src/autoencoder/config/familias.json"
else:
    FAM_J = RAIZ / "src/mfca_n1_autoencoder/familias.json"

SAL = Path(os.environ.get("DCSA_AE_OUTPUT_DIR", os.environ.get("DCSA_RESULTS_DIR", RAIZ / "results/DCSA_octubre/autoencoder")))

for sub in ("modelos", "tablas", "figuras", "manifiestos"):
    (SAL / sub).mkdir(parents=True, exist_ok=True)

CFG = json.loads(FAM_J.read_text(encoding="utf-8"))
FAM_ESC = [f for f in CFG["familias_escala"] if not f.get("analisis_usuario", False)]
FAM_PRE = CFG["familias_pregunta"]
TXT_COLS = [q["columna_texto"] for q in FAM_PRE]

def _sql(db_path: Path, query: str) -> pd.DataFrame:
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql(query, conn)

esc = _sql(DB_ESC, "SELECT * FROM medicion")
pun = _sql(DB_ESC, "SELECT * FROM puntaje")
emp = _sql(DB_ESC, "SELECT * FROM empalme_narrativa")
panel_raw = _sql(DB_NAR, "SELECT * FROM panel")

print(f"[TRAZABILIDAD] Fila inicial de tabla 'panel' en SQLite: {len(panel_raw)}")

# 1. Auditoría explícita de exclusiones canónicas
null_mask = panel_raw["id_participante"].isna() | panel_raw["semana_num"].isna() | panel_raw["fecha"].isna()
filas_nulas = panel_raw[null_mask]
print(f"[TRAZABILIDAD] Filas excluidas por nulos (N21 sem 10-11, N34): {len(filas_nulas)}")

panel = panel_raw.copy()
panel["id_participante"] = panel["id_participante"].astype(str).str.strip()
panel["fecha"] = pd.to_datetime(panel["fecha"], errors="coerce")
panel["semana_num"] = pd.to_numeric(panel["semana_num"], errors="coerce")
panel = panel.dropna(subset=["id_participante", "fecha", "semana_num"]).copy()
panel["semana_num"] = panel["semana_num"].astype(int)
panel = panel.sort_values(["id_participante", "semana_num", "fecha"]).reset_index(drop=True)
print(f"[TRAZABILIDAD] Filas tras depuración de nulos: {len(panel)} (447 reportadas históricamente)")

# 2. Descarte explícito de la semana 15 duplicada de N33
dups_sem = panel.duplicated(subset=["id_participante", "semana_num"]).sum()
if dups_sem > 0:
    print(f"[TRAZABILIDAD] Duplicadas detectadas en (id_participante, semana_num): {dups_sem} fila (N33 semana 15). Se conserva la primera.")
    panel = panel.drop_duplicates(subset=["id_participante", "semana_num"], keep="first").reset_index(drop=True)

print(f"[TRAZABILIDAD] Observaciones semanales únicas en SEM: {len(panel)} (coincide exactamente con las 446 evaluadas en diagnósticos)")
assert len(panel) == N_FILAS_ESPERADAS_SEM, f"Error: Se esperaban {N_FILAS_ESPERADAS_SEM} filas, halladas {len(panel)}"

panel["paso"] = panel.groupby("id_participante").cumcount() + 1

# 3. Pivot de puntajes y normalización teórica [0, 1]
PUN = pun.pivot_table(index="id_medicion", columns="escala", values="valor", aggfunc="first")
fams_presentes = [f["id"] for f in FAM_ESC if f["id"] in PUN.columns]
FAM_ESC = [f for f in FAM_ESC if f["id"] in fams_presentes]

F_df = esc[["id_medicion", "id_narrativa", "instrumento", "tiempo", "fecha"]].merge(
    PUN[fams_presentes].reset_index(), on="id_medicion", how="left"
)
for f in FAM_ESC:
    lo, hi = float(f["min_total"]), float(f["max_total"])
    vals = pd.to_numeric(F_df[f["id"]], errors="coerce")
    F_df["n_" + f["id"]] = ((vals - lo) / (hi - lo)).clip(lower=0.0, upper=1.0)

# 4. Empalme con ventana de ±7 días
emp_filtrado = emp[emp["distancia_dias"] <= 7].copy()
emp_filtrado["id_narrativa"] = emp_filtrado["id_narrativa"].astype(str).str.strip()
emp_filtrado["semana_narrativa"] = pd.to_numeric(emp_filtrado["semana_narrativa"], errors="coerce")

em = emp_filtrado.merge(
    F_df[["id_medicion"] + ["n_" + f["id"] for f in FAM_ESC]], on="id_medicion", how="left"
)
sem_join = panel.merge(
    em, left_on=["id_participante", "semana_num"], right_on=["id_narrativa", "semana_narrativa"], how="left"
)

cols_base_sem = ["id_participante", "semana_num", "fecha", "paso"] + TXT_COLS
if "indice_fuente" in panel.columns:
    cols_base_sem.append("indice_fuente")

SEM = panel[cols_base_sem].copy()
for f in FAM_ESC:
    col = "n_" + f["id"]
    d = sem_join.dropna(subset=[col]).sort_values("distancia_dias")
    v = d.drop_duplicates(subset=["id_participante", "semana_num"], keep="first").set_index(["id_participante", "semana_num"])[col]
    SEM = SEM.merge(v, on=["id_participante", "semana_num"], how="left")

SEM = SEM.sort_values(["id_participante", "paso"]).reset_index(drop=True)
SEM["id_fila_global"] = np.arange(len(SEM), dtype=np.int64)

PERS = sorted(SEM["id_participante"].unique())
PASO_MAX = int(SEM["paso"].max())
CLAVES_F = ["n_" + f["id"] for f in FAM_ESC]
cob_escala = SEM[CLAVES_F].notna().any(axis=1).mean()
print(f"[DATOS] Participantes: {len(PERS)} | Observaciones semanales: {len(SEM)} | Cobertura escalas: {100*cob_escala:.1f}%")


## 05 · Emparejamiento Exacto de Embeddings y Construcción de Tensores Multimodales


In [ ]:
# Ruta configurable para embeddings densos de 768 dimensiones
PQ_EMB_ENV = os.environ.get("DCSA_EMBEDDINGS_PATH")
if PQ_EMB_ENV and Path(PQ_EMB_ENV).is_file():
    PQ_EMB = Path(PQ_EMB_ENV).resolve()
else:
    PQ_EMB = Path(os.environ.get("DCSA_RESULTS_DIR", RAIZ / "results/DCSA_octubre")) / "embeddings_semanales.parquet"

if not PQ_EMB.is_file():
    raise FileNotFoundError(
        f"No se encontró el archivo de embeddings: '{PQ_EMB}'.\n"
        "Configure la variable de entorno DCSA_EMBEDDINGS_PATH con la ruta al archivo .parquet correspondiente."
    )

df_emb = pd.read_parquet(PQ_EMB)
COLS_EMB = [c for c in df_emb.columns if c.startswith("emb_")]
D_TXT = len(COLS_EMB)

df_emb = df_emb.copy()
df_emb["ID_Paciente"] = df_emb["ID_Paciente"].astype(str).str.strip()
df_emb["Num_Sesion"] = pd.to_numeric(df_emb["Num_Sesion"], errors="coerce").astype(int)
df_emb["Pregunta"] = df_emb["Pregunta"].astype(str).str.strip()

_agg = df_emb.groupby(["ID_Paciente", "Num_Sesion", "Pregunta"], as_index=False)[COLS_EMB].mean()
CLAVE_EMB = {
    (str(r.ID_Paciente), int(r.Num_Sesion), str(r.Pregunta)): np.asarray([getattr(r, c) for c in COLS_EMB], dtype=np.float32)
    for r in _agg.itertuples(index=False)
}

valores_pregunta_pq = set(_agg["Pregunta"].unique())
MAPA_CLAVE_PREG = {}
for idx_q, q in enumerate(FAM_PRE, start=1):
    candidatos_q = [
        str(q.get("columna_texto", "")).strip(), str(q.get("id", "")).strip(),
        str(q.get("numero", idx_q)).strip(), f"P{idx_q}", f"p{idx_q}", f"Q{idx_q}", f"q{idx_q}"
    ]
    coincidencias = [c for c in candidatos_q if c in valores_pregunta_pq]
    if not coincidencias:
        raise KeyError(f"Pregunta {q['id']} no hallada en embeddings_semanales.")
    MAPA_CLAVE_PREG[q["id"]] = coincidencias[0]

EMB = {}
MASC_EMB_FILA = {}
for q in FAM_PRE:
    qid = q["id"]
    clave_pq = MAPA_CLAVE_PREG[qid]
    M = np.zeros((len(SEM), D_TXT), dtype=np.float32)
    m_emb = np.zeros(len(SEM), dtype=np.float32)
    for fila in SEM.itertuples(index=False):
        pos = int(fila.id_fila_global)
        v = CLAVE_EMB.get((str(fila.id_participante), int(fila.semana_num), clave_pq))
        if v is not None and np.linalg.norm(v) > 1e-8:
            M[pos] = v
            m_emb[pos] = 1.0
    EMB[qid] = M
    MASC_EMB_FILA[qid] = m_emb

# Detección explícita de semanas sin texto (ej. N12 semana 6)
semanas_sin_texto = []
for fila in SEM.itertuples(index=False):
    pos = int(fila.id_fila_global)
    if sum(MASC_EMB_FILA[q["id"]][pos] for q in FAM_PRE) == 0:
        semanas_sin_texto.append((fila.id_participante, fila.semana_num))
print(f"[AUDITORÍA TEXTO] Semanas sin narrativa textual observada: {semanas_sin_texto}")

# Construcción de tensores longitudinales (N_PERS, PASO_MAX, ...)
N_PERS = len(PERS)
N_FAM = len(FAM_ESC)
N_PREG = len(FAM_PRE)

FAM_T = np.full((N_PERS, PASO_MAX, N_FAM), np.nan, dtype=np.float32)
TXT_T = np.zeros((N_PERS, PASO_MAX, N_PREG, D_TXT), dtype=np.float32)
PASO_T = np.zeros((N_PERS, PASO_MAX), dtype=np.float32)
MASC_T = np.zeros((N_PERS, PASO_MAX, N_PREG), dtype=np.float32)
SEMANA_NUM_T = np.full((N_PERS, PASO_MAX), -1, dtype=np.int32)
FILA_GLOBAL_T = np.full((N_PERS, PASO_MAX), -1, dtype=np.int64)

for pi, p in enumerate(PERS):
    s = SEM[SEM["id_participante"] == p].sort_values("paso").head(PASO_MAX)
    for j, fila in enumerate(s.itertuples(index=False)):
        pos = int(fila.id_fila_global)
        PASO_T[pi, j] = 1.0
        SEMANA_NUM_T[pi, j] = int(fila.semana_num)
        FILA_GLOBAL_T[pi, j] = pos
        for fi, c in enumerate(CLAVES_F):
            FAM_T[pi, j, fi] = getattr(fila, c)
        for qi, q in enumerate(FAM_PRE):
            qid = q["id"]
            TXT_T[pi, j, qi] = EMB[qid][pos]
            MASC_T[pi, j, qi] = MASC_EMB_FILA[qid][pos]

assert int(PASO_T.sum()) == len(SEM), "Biyección PASO_T vs SEM fallida."

MASC_F = np.isfinite(FAM_T).astype(np.float32)
FAM_T_CERO = np.nan_to_num(FAM_T, nan=0.0).astype(np.float32)

# Pares de semanas consecutivas reales (Delta semana == 1)
MASC_PARES_CONSEC = (
    (PASO_T[:, 1:] == 1.0) & (PASO_T[:, :-1] == 1.0) & ((SEMANA_NUM_T[:, 1:] - SEMANA_NUM_T[:, :-1]) == 1)
).astype(np.float32)

# Vector semántico semanal promedio normalizado
pesos_preg = MASC_T[..., None]
suma_emb = (TXT_T * pesos_preg).sum(axis=2)
conteo_preg = pesos_preg.sum(axis=2).clip(min=1.0)
ENTRADA_TXT = (suma_emb / conteo_preg).astype(np.float32)
norm_ent = np.linalg.norm(ENTRADA_TXT, axis=-1, keepdims=True).clip(min=1e-8)
ENTRADA_TXT_NORM = np.where(conteo_preg > 0, ENTRADA_TXT / norm_ent, 0.0).astype(np.float32)

# Entrada psicométrica: 7 puntajes normalizados [0, 1] + 7 máscaras binarias (14 dimensiones)
ENTRADA_PSI = np.concatenate([FAM_T_CERO, MASC_F], axis=-1).astype(np.float32)

print(f"[TENSORES] ENTRADA_TXT_NORM: {ENTRADA_TXT_NORM.shape} | ENTRADA_PSI: {ENTRADA_PSI.shape}")
print(f"[MÁSCARAS] Pasos activos: {int(PASO_T.sum())} | Escalas observadas: {int(MASC_F.sum())} | Preguntas con embedding: {int(MASC_T.sum())}")


## 06 · Entrenamiento con Partición Interna de Validación, Early Stopping y Restauración de Pesos


In [ ]:
def dividir_entrenamiento_validacion_interna(
    indices_tr: np.ndarray, fraccion_val: float = FRACCION_VAL_INT, semilla: int = SEMILLA
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(semilla)
    perm = rng.permutation(indices_tr)
    n_val = max(1, int(round(len(indices_tr) * fraccion_val)))
    val_int = np.sort(perm[:n_val])
    tr_int = np.sort(perm[n_val:])
    return tr_int, val_int

def entrenar_pliegue(
    idx_tr_ext: np.ndarray,
    pliegue_id: int,
    epocas_max: int = EPOCAS_MAX,
    lr: float = LR_INICIAL,
    modo_arq: str = MODO_ARQUITECTURA,
    d_latente: int = LATENTE,
    mu_txt_list: Optional[List[np.ndarray]] = None,
    verbose: bool = False,
) -> Tuple[nn.Module, List[Dict[str, Any]], Dict[str, Any]]:
    semilla_pliegue = SEMILLA + 100 * pliegue_id
    fijar_semillas(semilla_pliegue)

    tr_int, val_int = dividir_entrenamiento_validacion_interna(idx_tr_ext, semilla=semilla_pliegue)
    assert len(set(tr_int).intersection(set(val_int))) == 0, "Fuga tr_int vs val_int"

    # Cálculo estricto de medias de anclaje residual exclusivamente sobre tr_int (sin fuga de val_int)
    if mu_txt_list is None:
        mu_tr_int_list = []
        for qi in range(N_PREG):
            m_tr_q = MASC_T[tr_int][:, :, qi].astype(bool)
            mu_q = TXT_T[tr_int][:, :, qi, :][m_tr_q].mean(axis=0) if m_tr_q.any() else np.zeros(768, dtype=np.float32)
            mu_tr_int_list.append(mu_q)
    else:
        mu_tr_int_list = mu_txt_list

    t_dev = lambda a: torch.as_tensor(a, dtype=torch.float32, device=DEVICE)
    Xt_tr, Xp_tr = t_dev(ENTRADA_TXT_NORM[tr_int]), t_dev(ENTRADA_PSI[tr_int])
    F_tr, MF_tr = t_dev(FAM_T_CERO[tr_int]), t_dev(MASC_F[tr_int])
    T_tr, MT_tr = t_dev(TXT_T[tr_int]), t_dev(MASC_T[tr_int])
    MP_tr = t_dev(MASC_PARES_CONSEC[tr_int])

    Xt_va, Xp_va = t_dev(ENTRADA_TXT_NORM[val_int]), t_dev(ENTRADA_PSI[val_int])
    F_va, MF_va = t_dev(FAM_T_CERO[val_int]), t_dev(MASC_F[val_int])
    T_va, MT_va = t_dev(TXT_T[val_int]), t_dev(MASC_T[val_int])
    MP_va = t_dev(MASC_PARES_CONSEC[val_int])

    solo_txt = (modo_arq == "canonico_recalibrado")
    mod = AutoencoderMultimodalDualRecalibrado(
        d_in_txt=768, d_in_psi=14, d_latente=d_latente,
        nombres_fam=[f["id"] for f in FAM_ESC], nombres_preg=[q["id"] for q in FAM_PRE],
        solo_texto_entrada=solo_txt,
        mu_txt_list=mu_tr_int_list
    ).to(DEVICE)

    opt = torch.optim.AdamW(mod.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=6)

    rng_lotes = np.random.default_rng(semilla_pliegue)
    mejor_val_loss = float("inf")
    mejor_epoca = 1
    mejor_estado = {k: v.detach().cpu().clone() for k, v in mod.state_dict().items()}
    sin_mejora = 0
    historial: List[Dict[str, Any]] = []
    n_tr = len(tr_int)

    for ep in range(1, epocas_max + 1):
        mod.train()
        perm = rng_lotes.permutation(n_tr)
        acum_tr = 0.0

        for i in range(0, n_tr, LOTE):
            idx = perm[i : i + LOTE]
            opt.zero_grad()
            z_b, pf_b, pt_b = mod(Xt_tr[idx], Xp_tr[idx])
            info_l = calcular_perdidas_recalibradas(
                pf_b, F_tr[idx], MF_tr[idx], pt_b, T_tr[idx], MT_tr[idx],
                z_seq=z_b, masc_pares_consec=MP_tr[idx]
            )
            loss_b = info_l["loss_total"]
            loss_b.backward()
            torch.nn.utils.clip_grad_norm_(mod.parameters(), MAX_GRAD_NORM)
            opt.step()
            acum_tr += float(loss_b.detach().cpu().item()) * (len(idx) / float(n_tr))

        mod.eval()
        with torch.no_grad():
            z_v, pf_v, pt_v = mod(Xt_va, Xp_va)
            info_v = calcular_perdidas_recalibradas(
                pf_v, F_va, MF_va, pt_v, T_va, MT_va, z_seq=z_v, masc_pares_consec=MP_va
            )
            val_loss = float(info_v["loss_total"].detach().cpu().item())

        sched.step(val_loss)
        historial.append({"pliegue": pliegue_id, "epoca": ep, "train_loss": acum_tr, "val_loss": val_loss})

        if val_loss < mejor_val_loss - 1e-4:
            mejor_val_loss = val_loss
            mejor_epoca = ep
            mejor_estado = {k: v.detach().cpu().clone() for k, v in mod.state_dict().items()}
            sin_mejora = 0
        else:
            sin_mejora += 1
            if sin_mejora >= PACIENCIA_EARLY_STOP:
                break

    mod.load_state_dict(mejor_estado)
    meta = {
        "pliegue": pliegue_id, "semilla_pliegue": semilla_pliegue,
        "mejor_epoca": mejor_epoca, "epocas_ejecutadas": len(historial), "mejor_val_loss": mejor_val_loss
    }
    return mod, historial, meta


## 07 · Validación Cruzada Agrupada, Baselines Comparables y Diagnósticos Latentes


In [ ]:
def diagnosticar_espacio_latente(
    Z_obs: np.ndarray, X_txt_obs: np.ndarray, X_psi_obs: np.ndarray
) -> Dict[str, Any]:
    """
    Diagnósticos espectrales y de preservación geométrica rigurosa:
      - Rango efectivo espectral de Roy-Vetterli
      - Dimensiones activas (sigma > 1e-3)
      - Correlación interdimensional media absoluta (redundancia)
      - Preservación geométrica Textual: Pearson r(D_X_txt, D_Z)
      - Preservación geométrica Multimodal: Pearson r(D_X_multi, D_Z) sobre TODOS los pares del pliegue.
    """
    n_obs, d_lat = Z_obs.shape
    sd_dim = np.std(Z_obs, axis=0, ddof=1) if n_obs > 1 else np.zeros(d_lat)
    dims_activas = int((sd_dim > 1e-3).sum())

    Zc = Z_obs - np.mean(Z_obs, axis=0, keepdims=True)
    _, s_vals, _ = np.linalg.svd(Zc, full_matrices=False)
    suma_s = float(np.sum(s_vals))
    if suma_s > 1e-12:
        p = s_vals / suma_s
        p_pos = p[p > 1e-12]
        rango_efectivo = float(np.exp(-np.sum(p_pos * np.log(p_pos))))
        var_top5 = float(np.sum(s_vals[:5] ** 2) / np.sum(s_vals ** 2))
    else:
        rango_efectivo = 0.0
        var_top5 = np.nan

    if dims_activas >= 2:
        C = np.corrcoef(Z_obs[:, sd_dim > 1e-3], rowvar=False)
        off_diag = C[~np.eye(C.shape[0], dtype=bool)]
        corr_abs_media = float(np.nanmean(np.abs(off_diag)))
    else:
        corr_abs_media = np.nan

    iu = np.triu_indices(n_obs, k=1)
    dist_z = np.sqrt(np.maximum(0.0, ((Z_obs[:, None, :] - Z_obs[None, :, :]) ** 2).sum(axis=-1)))
    dist_xt = np.sqrt(np.maximum(0.0, ((X_txt_obs[:, None, :] - X_txt_obs[None, :, :]) ** 2).sum(axis=-1)))

    if np.std(dist_z[iu]) > 1e-12 and np.std(dist_xt[iu]) > 1e-12:
        r_geom_txt = float(np.corrcoef(dist_xt[iu], dist_z[iu])[0, 1])
    else:
        r_geom_txt = np.nan

    dist_xp = np.sqrt(np.maximum(0.0, ((X_psi_obs[:, None, :] - X_psi_obs[None, :, :]) ** 2).sum(axis=-1)))
    dist_multi = np.sqrt(dist_xt ** 2 + dist_xp ** 2)
    if np.std(dist_z[iu]) > 1e-12 and np.std(dist_multi[iu]) > 1e-12:
        r_geom_multi = float(np.corrcoef(dist_multi[iu], dist_z[iu])[0, 1])
    else:
        r_geom_multi = np.nan

    return {
        "n_observaciones": int(n_obs),
        "d_latente": int(d_lat),
        "dims_activas_sd_gt_1e3": dims_activas,
        "rango_efectivo_roy_vetterli": rango_efectivo,
        "fraccion_var_top5_sv": var_top5,
        "corr_abs_media_interdim": corr_abs_media,
        "preservacion_geom_txt_r": r_geom_txt,
        "preservacion_geom_multi_r": r_geom_multi,
    }


GRUPOS = np.array(PERS)
gkf = GroupKFold(n_splits=PLIEGUES_EXT)
resultados_cv = []
historiales_cv = []
meta_pliegues = []
diagnosticos_latentes_cv = []

Z_OOF = np.full((N_PERS, PASO_MAX, LATENTE), np.nan, dtype=np.float32)

for k, (tr, te) in enumerate(gkf.split(ENTRADA_TXT_NORM, groups=GRUPOS), start=1):
    assert len(set(GRUPOS[tr]).intersection(set(GRUPOS[te]))) == 0, f"Fuga de participantes en pliegue {k}"

    # Calcular medias de entrenamiento por pregunta para anclaje neutral del cabezal residual
    mu_tr_list = []
    for qi in range(N_PREG):
        m_tr_q = MASC_T[tr][:, :, qi].astype(bool)
        mu_q = TXT_T[tr][:, :, qi, :][m_tr_q].mean(axis=0) if m_tr_q.any() else np.zeros(768, dtype=np.float32)
        mu_tr_list.append(mu_q)

    # mu_txt_list se delega a entrenar_pliegue para calcularse exclusivamente sobre tr_int (sin fuga)
    mod_k, hist_k, meta_k = entrenar_pliegue(tr, pliegue_id=k, mu_txt_list=None, verbose=(k == 1))
    historiales_cv.extend(hist_k)
    meta_pliegues.append(meta_k)

    ruta_modelo_k = SAL / "modelos" / f"ae_recalibrado_focalizado_pliegue_{k}_{MARCA_EJECUCION}.pt"
    torch.save({"pliegue": k, "state_dict": mod_k.state_dict(), "meta": meta_k}, ruta_modelo_k)

    mod_k.eval()
    with torch.no_grad():
        x_te_t = torch.tensor(ENTRADA_TXT_NORM[te], dtype=torch.float32, device=DEVICE)
        x_te_p = torch.tensor(ENTRADA_PSI[te], dtype=torch.float32, device=DEVICE)
        z_te, cf_te, ct_te = mod_k(x_te_t, x_te_p)

    zz = z_te.cpu().numpy()
    cf = cf_te.cpu().numpy()
    ct = ct_te.cpu().numpy()
    Z_OOF[te] = zz

    masc_paso_te = PASO_T[te].astype(bool)
    diag_k = diagnosticar_espacio_latente(
        zz[masc_paso_te], ENTRADA_TXT_NORM[te][masc_paso_te], ENTRADA_PSI[te][masc_paso_te][:, :7]
    )
    diag_k["pliegue"] = k
    diagnosticos_latentes_cv.append(diag_k)

    # 1. Evaluación de las 7 cabezas psicométricas
    for fi, f in enumerate(FAM_ESC):
        m = MASC_F[te][:, :, fi].astype(bool)
        m_tr = MASC_F[tr][:, :, fi].astype(bool)
        mu_tr = float(FAM_T[tr][:, :, fi][m_tr].mean()) if m_tr.any() else 0.5

        if m.sum() > 0:
            y_r, y_p = FAM_T[te][:, :, fi][m], cf[:, :, fi][m]
            mse_mod = float(np.mean((y_r - y_p) ** 2))
            mae_mod = float(np.mean(np.abs(y_r - y_p)))
            var_r = float(np.var(y_r))
            r2_mod = float(1.0 - mse_mod / var_r) if var_r > 1e-12 else np.nan
            mse_base = float(np.mean((y_r - mu_tr) ** 2))
        else:
            mse_mod = mae_mod = r2_mod = mse_base = np.nan

        resultados_cv.append({
            "pliegue": k, "cabeza": f["id"], "tipo": "escala", "n_obs": int(m.sum()),
            "mse": mse_mod, "mae": mae_mod, "r2": r2_mod, "r2_global": r2_mod, "cos_sim": np.nan,
            "base_media_mse": mse_base, "base_pca_mse": np.nan,
            "gana_a_la_media": bool(mse_mod < mse_base)
        })

    # 2. Evaluación de los 8 cabezales semánticos de preguntas
    for qi, q in enumerate(FAM_PRE):
        m = MASC_T[te][:, :, qi].astype(bool)
        m_tr = MASC_T[tr][:, :, qi].astype(bool)
        mu_tr_vec = TXT_T[tr][:, :, qi, :][m_tr].mean(axis=0) if m_tr.any() else np.zeros(768, dtype=np.float32)

        if m.sum() > 0:
            Y_r, Y_p = TXT_T[te][:, :, qi, :][m], ct[:, :, qi, :][m]
            mse_mod = float(np.mean((Y_r - Y_p) ** 2))
            mae_mod = float(np.mean(np.abs(Y_r - Y_p)))
            var_total = float(np.mean((Y_r - mu_tr_vec) ** 2))
            r2_glob = float(1.0 - mse_mod / var_total) if var_total > 1e-12 else np.nan
            
            var_c = np.var(Y_r, axis=0)
            mse_c = np.mean((Y_r - Y_p) ** 2, axis=0)
            r2_mod = float(np.mean(1.0 - mse_c[var_c > 1e-12] / var_c[var_c > 1e-12]))
            
            norm_r = np.linalg.norm(Y_r, axis=-1).clip(min=1e-8)
            norm_p = np.linalg.norm(Y_p, axis=-1).clip(min=1e-8)
            cos_mod = float(np.mean(np.sum(Y_r * Y_p, axis=-1) / (norm_r * norm_p)))

            # Baseline de la media
            mse_base = float(np.mean((Y_r - mu_tr_vec) ** 2))

            # Baseline PCA (32 componentes ajustados en train)
            pca = PCA(n_components=min(32, int(m_tr.sum()) - 1), random_state=SEMILLA)
            pca.fit(TXT_T[tr][:, :, qi, :][m_tr])
            rec_pca = pca.inverse_transform(pca.transform(Y_r))
            mse_pca = float(np.mean((Y_r - rec_pca) ** 2))
        else:
            mse_mod = mae_mod = r2_mod = r2_glob = cos_mod = mse_base = mse_pca = np.nan

        resultados_cv.append({
            "pliegue": k, "cabeza": "txt::" + q["id"], "tipo": "pregunta", "n_obs": int(m.sum()),
            "mse": mse_mod, "mae": mae_mod, "r2": r2_mod, "r2_global": r2_glob, "cos_sim": cos_mod,
            "base_media_mse": mse_base, "base_pca_mse": mse_pca,
            "gana_a_la_media": bool(mse_mod < mse_base)
        })

    print(f"[CV] Pliegue {k}/{PLIEGUES_EXT} completado | Sujetos test={len(te)} | "
          f"Rango_Z={diag_k['rango_efectivo_roy_vetterli']:.2f} | r_geom_multi={diag_k['preservacion_geom_multi_r']:.3f}")

RES_DF = pd.DataFrame(resultados_cv)
DIAG_DF = pd.DataFrame(diagnosticos_latentes_cv)

RESU = RES_DF.groupby(["cabeza", "tipo"]).agg(
    n_obs=("n_obs", "sum"),
    mse=("mse", "mean"),
    mae=("mae", "mean"),
    r2=("r2", "mean"),
    r2_global=("r2_global", "mean"),
    cos_sim=("cos_sim", "mean"),
    base_media_mse=("base_media_mse", "mean"),
    base_pca_mse=("base_pca_mse", "mean"),
    tasa_gana_media=("gana_a_la_media", "mean")
).reset_index()
RESU["mejora_sobre_media_pct"] = (100.0 * (1.0 - RESU["mse"] / RESU["base_media_mse"])).round(2)

print("\n=== RESUMEN RECALIBRADO FOCALIZADO POR CABEZAL (OUT-OF-FOLD) ===")
print(RESU.to_string(index=False))

print("\n=== DIAGNÓSTICOS DEL ESPACIO LATENTE POR PLIEGUE ===")
print(DIAG_DF.to_string(index=False))


## 08 · Generación y Verificación Estricta de `ae_presente_latents.parquet` (`446 × (4 + d_lat)`)


In [ ]:
# Construcción de la matriz latente Out-of-Fold respetando el orden estricto de SEM
masc_activos = (PASO_T == 1.0)
filas_activas = FILA_GLOBAL_T[masc_activos]
z_activos = Z_OOF[masc_activos]

assert len(filas_activas) == len(SEM), "Discrepancia entre pasos activos y SEM."
orden_sem = np.argsort(filas_activas)
Z_ORDENADO = z_activos[orden_sem]

cols_lat = [f"lat_{i:02d}" for i in range(LATENTE)]
DF_LAT = pd.DataFrame(Z_ORDENADO, columns=cols_lat)
DF_LAT.insert(0, "paso", SEM["paso"].values)
DF_LAT.insert(0, "fecha", SEM["fecha"].astype(str).values)
DF_LAT.insert(0, "semana_num", SEM["semana_num"].values)
DF_LAT.insert(0, "id_participante", SEM["id_participante"].values)

assert len(DF_LAT) == N_FILAS_ESPERADAS_SEM, f"Filas inválidas en DF_LAT: {len(DF_LAT)}"
assert not DF_LAT.isna().any().any(), "Valores NaN detectados en la matriz latente generada."

# Guardar matriz latente en su única ubicación canónica
ruta_pq_latents = SAL / "modelos" / "ae_presente_latents.parquet"
DF_LAT.to_parquet(ruta_pq_latents, index=False)

print(f"[EXPORTACIÓN] ae_presente_latents.parquet generado con éxito en destino canónico: {DF_LAT.shape} -> {ruta_pq_latents}")


## 09 · Indicadores de Sistemas Complejos: `_z_intra` con Winsorizado Real, Polaridad de IRC_D y Rejilla Continua


In [ ]:
MIN_PASOS_IRC = 8

def _z_intra(s: pd.Series, umbral: float = 3.0) -> pd.Series:
    """
    Normalización intra-sujeto con RECORTADO GENUINO (clipping) al umbral,
    preservando la magnitud y el sentido de las crisis agudas en lugar de anularlas a 0.0.
    """
    v = pd.to_numeric(s, errors="coerce")
    val = v.dropna()
    if len(val) < 3:
        return pd.Series(np.nan, index=s.index)
    std = val.std(ddof=1)
    if std < 1e-8 or not np.isfinite(std):
        return pd.Series(0.0, index=s.index)
    z = (v - val.mean()) / std
    return z.clip(lower=-umbral, upper=umbral)

def autocorr_lag1_rejilla(serie_valores: pd.Series, serie_semanas: pd.Series) -> float:
    """Calcula la autocorrelación lag-1 estricta sobre semanas consecutivas reales (Delta semana == 1)."""
    df_temp = pd.DataFrame({"semana": serie_semanas, "val": serie_valores}).dropna()
    if len(df_temp) < 4:
        return np.nan
    s_min, s_max = int(df_temp["semana"].min()), int(df_temp["semana"].max())
    rejilla = pd.DataFrame({"semana": np.arange(s_min, s_max + 1)})
    mrg = rejilla.merge(df_temp, on="semana", how="left")
    pares = pd.DataFrame({"y_t": mrg["val"].iloc[1:].values, "y_lag1": mrg["val"].iloc[:-1].values}).dropna()
    if len(pares) < 4 or pares["y_lag1"].std() < 1e-8 or pares["y_t"].std() < 1e-8:
        return np.nan
    return float(pares["y_lag1"].corr(pares["y_t"]))

# Cálculo por participante
filas_irc = []
for p in PERS:
    sp = SEM[SEM["id_participante"] == p].sort_values("semana_num")
    if len(sp) < MIN_PASOS_IRC:
        continue
    # Inversión de activación para eje de desregulación conductual
    z_evit = _z_intra(sp["n_bads_evitacion_rumia"])
    z_inact = -_z_intra(sp["n_bads_activacion"])
    eje_cond = 0.5 * (z_evit + z_inact)
    ar1_cond = autocorr_lag1_rejilla(eje_cond, sp["semana_num"])

    filas_irc.append({
        "id_participante": p, "n_semanas": len(sp), "ar1_desregulacion_conductual": ar1_cond
    })

DF_IRC = pd.DataFrame(filas_irc)
print(f"[SISTEMAS COMPLEJOS] IRC_D calculado para {len(DF_IRC)} participantes elegibles.")


## 10 · Figuras Diagnósticas, Volcado Reproducible y Manifiesto de Integridad SHA-256


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
df_hist = pd.DataFrame(historiales_cv)
for k in df_hist["pliegue"].unique():
    h_k = df_hist[df_hist["pliegue"] == k]
    plt.plot(h_k["epoca"], h_k["val_loss"], alpha=0.7, label=f"Pliegue {k}")
plt.xlabel("Época")
plt.ylabel("Pérdida Validación Interna")
plt.title("Convergencia y Early Stopping")
plt.legend()

plt.subplot(1, 2, 2)
bars = plt.bar(RESU["cabeza"], RESU["mejora_sobre_media_pct"])
plt.axhline(0, color="black", linestyle="--")
plt.xticks(rotation=90)
plt.ylabel("Mejora sobre Baseline de Media (%)")
plt.title("Ganancia Predictiva por Cabezal")
plt.tight_layout()

fig_path = SAL / "figuras" / f"fig_diagnosticos_recalibrado_{MARCA_EJECUCION}.png"
plt.savefig(fig_path, dpi=150)
plt.close()

# Volcado de métricas en tablas/
RES_DF.to_csv(SAL / "tablas" / f"metricas_por_cabeza_pliegue_{MARCA_EJECUCION}.csv", index=False)
RESU.to_csv(SAL / "tablas" / f"resumen_por_cabeza_{MARCA_EJECUCION}.csv", index=False)
DIAG_DF.to_csv(SAL / "tablas" / f"diagnosticos_espacio_latente_{MARCA_EJECUCION}.csv", index=False)

# Manifiesto SHA-256 en manifiestos/
manifiesto = {
    "marca": MARCA_EJECUCION,
    "modo": MODO,
    "modo_arquitectura": MODO_ARQUITECTURA,
    "d_latente": LATENTE,
    "sha256_latents": sha256_archivo(ruta_pq_latents),
    "dimensiones_latents": list(DF_LAT.shape),
}
with open(SAL / "manifiestos" / f"manifiesto_integridad_ae_{MARCA_EJECUCION}.json", "w") as f:
    json.dump(manifiesto, f, indent=2)

print(f"[REPRODUCIBILIDAD] Salidas guardadas y verificadas con SHA-256: {manifiesto['sha256_latents'][:16]}...")
